# Ashworth et al. (2025), Poisson Gaussian-process fits of ongoing activity, on dandiset 001708

**Paper.** Ashworth et al., "Multi-timescale dynamics organize descending pain modulation", bioRxiv
doi:10.64898/2025.12.15.693905, version 4, with its supplement; the claims file
(`claims/001708/10.64898-2025.12.15.693905.yaml`) was written from this text. Method details come from
the paper or its code (github.com/nox-lab/multi-timescale-rhythmic-dynamics-in-rvm-neurons at commit
e12f3ee771f4b5506e5db0bac09762ba6044a4d6), cited where used.

**Dataset.** DANDI:001708, version 0.260604.1023 (cited by the paper; the only published version), streamed.

| claim | tested here |
|---|---|
| fig5b-train-r2 | yes |
| fig5b-train-no-difference | yes |
| fig5b-test-r2 | yes |
| fig5b-test-group-differences | yes |
| table1-hyperparameters | yes |
| fig5c-dispersion | yes |
| table1-latent-variance-order | yes |
| fig5d-phase-profiles | yes |
| figs2-phase-latency | yes |

**What is run.** The model of the Methods (Gaussian process modeling, paragraph 2) as the code
implements it (non_poisson_gpytorch_model.py, Poisson_GP_main_fit_notebook.ipynb): 5 s bins, the
last 300 s of each cell held out, sparse variational GP with fixed evenly spaced inducing points,
constant mean, scaled periodic kernel (period initialized at 300 s, lengthscale at 0.1 with the
prior N(0.1, 0.1) on the raw lengthscale), rate 10 softplus(f), generalized Poisson likelihood with
alpha < 1 and prior N(0, 0.5), uniform contamination over 0 to 200 counts, Adam with learning rate
0.05, seed 0; and the constant-mean null GP of the code, for the pseudo-R2.

Settings the criterion names that could not all be run in the notebook limit (timed on 50 to 100
iterations while preparing, not run here): the code's 100 inducing points and p_noise 0.01 are
used. The Methods' 200 inducing points and p_noise 0.001 would take about 0.55 s per iteration
for the batch, about 90 min at 10000 iterations, over the 60 min limit. The number of iterations
is 5000, the length of every loss history the code saved (loss_histories.csv); params.json gives
10000, which would bring this notebook near 60 min. Results are also shown at 2500 iterations,
to show how far they still move. The Methods' 200 inducing points and p_noise 0.001 are run for 2500
iterations in gp_methods.ipynb.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import torch
import json
import rvm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
T0 = time.time()
print("dandiset", rvm.DANDISET, "version", rvm.VERSION, "torch", torch.__version__)

## Data check

In [ ]:
sessions, cells = rvm.load_all()
print(f"loaded {len(sessions)} files in {time.time() - T0:.0f} s")
ong = cells[cells.protocol.isin(["ongoing", "evoked/ongoing"])].copy()
print(pd.crosstab(ong.protocol, ong.cls, margins=True))
print("subjects per class:", ong.groupby("cls").subject.nunique().to_dict(), "; files per class:", ong.groupby("cls").path.nunique().to_dict())
SEG = {}
for ci, c in ong.iterrows():
    if c.protocol == "ongoing":
        a, b, gap, buf = rvm.ongoing_segment(sessions[c.session], c.t_stop, 1500.0)
        SEG[ci] = (a, b)
print("ongoing-file cells with a 1500 s segment:", len(SEG), "; segment starts >= 0:", all(a >= 0 for a, b in SEG.values()))

# Inputs per cell. Main reading (the criterion): ON and OFF 1500 s segments split at 1200 s;
# evoked/ongoing NEUTRAL cells, 960 s, split at 660 s (the notebook); ongoing NEUTRAL cells over
# their whole recording less its last 300 s (Methods, Data processing: "the full recording period
# from all neutral cells was used for GP modeling"). Variant: ongoing NEUTRAL cells on the code's
# 1500 s segment split at 1200 s.
XY, META = [], []
for ci, c in ong.iterrows():
    tr = sessions[c.session]["trains"][c.unit]
    if c.protocol == "evoked/ongoing":
        XY.append(rvm.bin_split(tr, 960.0, 660.0)); META.append((ci, "main"))
    else:
        a, b = SEG[ci]
        seg_xy = rvm.bin_split(rvm.cut(tr, a, b), 1500.0, 1200.0)
        if c.cls == "NEUTRAL":
            XY.append(seg_xy); META.append((ci, "code_segment"))
        else:
            XY.append(seg_xy); META.append((ci, "main"))
XY_full, META_full = [], []
for ci, c in ong[(ong.protocol == "ongoing") & (ong.cls == "NEUTRAL")].iterrows():
    tr = sessions[c.session]["trains"][c.unit]
    XY_full.append(rvm.bin_split(tr, c.t_stop, c.t_stop - 300.0)); META_full.append((ci, "main"))
lens = pd.DataFrame([dict(cls=cells.loc[ci, "cls"], kind=k, train_bins=len(xy[1]), test_bins=len(xy[3]), max_count=int(xy[1].max()))
                     for (ci, k), xy in zip(META + META_full, XY + XY_full)])
print(lens.groupby(["cls", "kind"]).agg(["min", "max"]))

## Fits

In [ ]:
torch.set_num_threads(4)
IT2, IT = 2500, 5000  # iterations: the snapshot and the end of the run
t1 = time.time()
FIT_A = rvm.fit_gp_fast(XY, n_iter=IT, num_inducing=100, p_noise=0.01, lr=0.05, seed=0, snapshots=(IT2,), null_same=True)
print(f"batch of {len(XY)} cells: {time.time() - t1:.0f} s")
t1 = time.time()
FIT_B = rvm.fit_gp_fast(XY_full, n_iter=IT, num_inducing=100, p_noise=0.01, lr=0.05, seed=0, snapshots=(IT2,), null_same=True)
print(f"batch of {len(XY_full)} ongoing NEUTRAL cells on whole recordings: {time.time() - t1:.0f} s")

def table(it, neutral="main"):
    rows = []
    for (ci, kind), j in zip(META, range(len(META))):
        if kind == "code_segment" and neutral == "main":
            continue
        r = FIT_A[it]
        rows.append(dict(ci=ci, kind=kind, j=j, batch="A", **{k: float(r[k][j]) for k in ["period", "lengthscale", "outputscale", "alpha", "train_r2", "test_r2", "train_r2_same", "test_r2_same", "train_null_ell", "test_null_ell"]}))
    if neutral == "main":
        for (ci, kind), j in zip(META_full, range(len(META_full))):
            r = FIT_B[it]
            rows.append(dict(ci=ci, kind=kind, j=j, batch="B", **{k: float(r[k][j]) for k in ["period", "lengthscale", "outputscale", "alpha", "train_r2", "test_r2", "train_r2_same", "test_r2_same", "train_null_ell", "test_null_ell"]}))
    d = pd.DataFrame(rows)
    d["cls"] = cells.loc[d.ci, "cls"].values
    d["subject"] = cells.loc[d.ci, "subject"].values
    d["file"] = cells.loc[d.ci, "path"].values
    return d

TAB = {(it, nr): table(it, nr) for it in (IT2, IT) for nr in ("main", "code")}
D = TAB[(IT, "main")]
print("cells per class, main reading:", D.cls.value_counts().to_dict())
print("non-finite values:", int((~np.isfinite(D[["period", "lengthscale", "outputscale", "alpha", "train_r2", "test_r2"]])).sum().sum()))
for key, d in TAB.items():
    print(key, d.groupby("cls")[["train_r2", "test_r2", "period", "alpha", "outputscale"]].median().round(4).to_dict("index"))

In [ ]:
ref = pd.read_csv(rvm.code_file("notebooks/gpytorch_models/results/gen_pois_no_exp/combined_data.csv"))
ref["cls"] = ref.cell_name.str.replace("_EXTRA", "")
print("the code's stored fits (combined_data.csv), class medians:")
print(ref.groupby("cls")[["Train Pseudo-R2", "Test Pseudo-R2", "period_length", "lengthscale", "outputscale", "w"]].median().round(4))
print("noise_prob (p_noise) of the stored fits:", sorted(ref.noise_prob.unique()))
code_nb = rvm.code_file("notebooks/gpytorch_models/Poisson_GP_main_fit_notebook.ipynb").read().decode()
print("the code creates the null model with:",
      sorted({ln.strip() for c in json.loads(code_nb)["cells"] for ln in "".join(c["source"]).splitlines() if "create_mean_model(T_max" in ln}),
      "(uniform contamination over 0 to the cell's largest training count; the periodic model's runs to max_count=200)")
print("rows per class:", ref.cls.value_counts().to_dict(), "; distinct (period, w) rows per class:",
      ref.groupby("cls").apply(lambda d: d[["period_length", "w"]].round(6).drop_duplicates().shape[0]).to_dict())
# per-cell comparison: the cells fitted here (code's settings, NEUTRAL on the code's segments) matched to the stored rows
MATCH = rvm.match_to_code(TAB[(IT, "code")], ref)
print("matched pairs per class:", MATCH.cls.value_counts().to_dict(), "; median relative difference of the null log likelihoods:",
      MATCH.groupby("cls").cost.median().round(4).to_dict(), "; largest:", MATCH.groupby("cls").cost.max().round(4).to_dict())
for v, cv in [("period", "code_period"), ("outputscale", "code_outputscale"), ("alpha", "code_alpha"), ("train_r2", "code_train_r2"), ("test_r2", "code_test_r2")]:
    print(f"  {v}: Spearman r by class", {k: round(float(stats.spearmanr(g[v], g[cv])[0]), 3) for k, g in MATCH.groupby("cls")},
          "; median absolute difference", MATCH.assign(d=(MATCH[v] - MATCH[cv]).abs()).groupby("cls").d.median().round(4).to_dict())
# which NEUTRAL input the stored rows come from: the 12 stored NEUTRAL rows of the ongoing files matched to the fits here
# on the code's 1500 s segments and on the whole recordings (the criterion's main reading); the 20 evoked/ongoing
# NEUTRAL cells (960 s) are in both sets, so only the ongoing-file cells are compared
ong_neut = set(ong[(ong.protocol == "ongoing") & (ong.cls == "NEUTRAL")].index)
for key, lab in [((IT, "code"), "code's 1500 s segments"), ((IT, "main"), "whole recordings")]:
    m = rvm.match_to_code(TAB[key][TAB[key].cls == "NEUTRAL"], ref)
    m = m[m.ci.isin(ong_neut)]
    print(f"NEUTRAL ongoing-file cells on the {lab}: {len(m)} matched; relative difference of the null log likelihoods "
          f"median {m.cost.median():.4f}, largest {m.cost.max():.4f}; train pseudo-R2 median here {m.train_r2.median():.4f}, "
          f"stored {m.code_train_r2.median():.4f}; Spearman r of train pseudo-R2 {stats.spearmanr(m.train_r2, m.code_train_r2)[0]:.3f}")


## Tests (bootstrap_R2_across_animals.ipynb of the code)

In [ ]:
# The tests (signflip, mw_blockwise, kw_blockwise, mw_within_file, r2_tests, group_tests) are in the
# helper, as bootstrap_R2_across_animals.ipynb of the code runs them; gp_methods.ipynb uses them too.
from rvm import signflip, mw_blockwise, kw_blockwise, mw_within_file, r2_tests, group_tests

## fig5b-train-r2

> A Poisson-type periodic Gaussian process fits the training data of all three cell classes with
> significant accuracy (pseudo-R2 different from 0; ON p = 0.0047, OFF p = 0.0004, NEUTRAL p < 0.0001).

In [ ]:
r2_tests(TAB[(IT, "main")], "train_r2", "5000 iterations, main reading")
r2_tests(TAB[(IT, "code")], "train_r2", "5000 iterations, ongoing NEUTRAL on the code's 1500 s segments")
r2_tests(TAB[(IT2, "main")], "train_r2", "2500 iterations, main reading")
r2_tests(TAB[(IT, "main")], "train_r2_same", "5000 iterations, main reading, null model with the periodic model's contamination range")
print("paper (combined_data.csv) medians: ON 0.0448, OFF 0.0479, NEUTRAL 0.0281")

**Result: reproduced, resting on an assumption.** Median training pseudo-R2 0.0478, 0.0497 and 0.0216 (ON, OFF, NEUTRAL), sign-flip p by animal 0.0079, 0.0021 and 0.0001. With the null model's contamination over 0 to 200 the medians are 0.0519, 0.0533 and 0.0249 with the same p.

## fig5b-train-no-difference

> Training pseudo-R2 does not differ between the three classes (permuted, Bonferroni-corrected
> Mann-Whitney U tests; Kruskal-Wallis p = 0.117).

In [ ]:
G_train = group_tests(TAB[(IT, "main")], "train_r2", "train pseudo-R2, 5000 iterations, main reading")
group_tests(TAB[(IT, "code")], "train_r2", "train pseudo-R2, NEUTRAL on the code's segments")
group_tests(TAB[(IT, "main")], "train_r2_same", "train pseudo-R2, main reading, null model with the periodic model's contamination range")
group_tests(TAB[(IT2, "main")], "train_r2", "train pseudo-R2, 2500 iterations, main reading")

**Result: not reproduced, resting on an assumption.** On the criterion's main reading (blocks by file, ongoing NEUTRAL cells over their whole recordings) ON against NEUTRAL differs (Bonferroni p 0.036) and Kruskal-Wallis gives p 0.034. Several variants find no difference: blocks by animal (Bonferroni 0.111, KW 0.075), the code's 1500 s NEUTRAL segments (Bonferroni 0.108, KW 0.098 by file; 0.270 and 0.204 by animal), and 2500 iterations with blocks by animal (0.075, 0.058). With the null's contamination over 0 to 200, blocks by file (Bonferroni 0.015, KW 0.018) and blocks by animal (KW 0.038), and at 2500 iterations with blocks by file (0.024, 0.028), a difference is found. The stored NEUTRAL fits of the ongoing files match the fits here on the code's 1500 s segments (median relative difference of the null log likelihoods 0.0003, Spearman r of the training pseudo-R2 0.993) and not those on whole recordings (1.28, r 0.112), so the paper's NEUTRAL inputs were the segments, on which no difference is found; this is asked of a reader (questions/001708.yaml).


## fig5b-test-r2

> On held-out data ON cells generalize significantly (p = 0.0067), with most cells above 0; OFF cells as
> a population do not (p = 0.3127), mixing well and poorly predicted cells; NEUTRAL cells fall
> significantly below baseline (p = 0.0033).

In [ ]:
r2_tests(TAB[(IT, "main")], "test_r2", "5000 iterations, main reading")
r2_tests(TAB[(IT, "code")], "test_r2", "5000 iterations, ongoing NEUTRAL on the code's 1500 s segments")
r2_tests(TAB[(IT2, "main")], "test_r2", "2500 iterations, main reading")
r2_tests(TAB[(IT, "main")], "test_r2_same", "5000 iterations, main reading, null model with the periodic model's contamination range")
print("paper (combined_data.csv) medians: ON 0.0372, OFF 0.0187, NEUTRAL -0.0128")

**Result: reproduced, resting on an assumption.** ON generalizes (median 0.0406, 21 of 25 above 0, p 0.0079), OFF does not (median 0.0197, p 0.373), NEUTRAL falls below baseline (median -0.0108, p 0.012). With the null's contamination over 0 to 200 the NEUTRAL median is -0.0049 and p 0.052, so that part would fail.

## fig5b-test-group-differences

> Test pseudo-R2 of NEUTRAL cells differs from that of ON cells (p < 0.001) and OFF cells (p = 0.006),
> ON and OFF cells do not differ (p = 0.939), and the Kruskal-Wallis test across the three is
> significant at test time (p < 0.001).

In [ ]:
G_test = group_tests(TAB[(IT, "main")], "test_r2", "test pseudo-R2, 5000 iterations, main reading")
group_tests(TAB[(IT, "code")], "test_r2", "test pseudo-R2, NEUTRAL on the code's segments")
group_tests(TAB[(IT, "main")], "test_r2_same", "test pseudo-R2, main reading, null model with the periodic model's contamination range")
print("--- variant: labels shuffled within each file (clustered_mannwhitney_u)")
for g1, g2 in [("ON", "OFF"), ("ON", "NEUTRAL"), ("OFF", "NEUTRAL")]:
    u, p = mw_within_file(TAB[(IT, "main")], "test_r2", g1, g2)
    print(f"  {g1} vs {g2}: U {u:.0f}, p {p:.4f}")

**Result: reproduced, resting on an assumption.** NEUTRAL differs from ON (Bonferroni p below 0.001) and OFF (0.015), ON and OFF do not (0.79), and Kruskal-Wallis gives p 0.001. With the null's contamination over 0 to 200 the corrected p are below 0.001, 0.021 and 0.63, and Kruskal-Wallis below 0.001.

## table1-hyperparameters, table1-latent-variance-order, fig5c-dispersion

> table1-hyperparameters: The fitted GP periods of ON and OFF cells cluster near 300 s (ON 297 ± 22 s,
> OFF 304 ± 27 s); NEUTRAL cells give 322 ± 54 s, which the paper reads as a weak constraint given their
> low latent variance and poor prediction.

> table1-latent-variance-order: The latent GP variance separates the classes; ON cells have the
> largest modulation amplitude (outputscale 17 ± 26), OFF cells intermediate (3.1 ± 4) and NEUTRAL
> cells a near-flat latent structure (0.1 ± 0.2).

> fig5c-dispersion: ON and OFF cells have positive dispersion parameters (ON 0.63 ± 0.25, p = 0.0011;
> OFF 0.36 ± 0.53, p = 0.0105), consistent with bursty over-dispersed firing, while NEUTRAL cells have
> negative dispersion (-0.9 ± 0.63, p < 0.001), firing more regularly than Poisson.

In [ ]:
def table1(d, label):
    t = d.groupby("cls")[["period", "lengthscale", "outputscale", "alpha"]].agg(["mean", "std", "median"])
    print(f"--- {label}")
    print(t.round(3))
    return t
T1 = table1(TAB[(IT, "main")], "5000 iterations, main reading")
table1(TAB[(IT, "code")], "5000 iterations, NEUTRAL on the code's segments")
table1(TAB[(IT2, "main")], "2500 iterations, main reading")
exp_p = {"ON": (297, 22), "OFF": (304, 27), "NEUTRAL": (322, 54)}
exp_l = {"ON": (0.73, 0.018), "OFF": (0.74, 0.013), "NEUTRAL": (0.74, 0.0068)}
for k in ["ON", "OFF", "NEUTRAL"]:
    m, l = T1.loc[k, ("period", "mean")], T1.loc[k, ("lengthscale", "mean")]
    print(f"{k}: period {m:.1f} vs {exp_p[k][0]} +- {exp_p[k][1]} (within: {abs(m - exp_p[k][0]) <= exp_p[k][1]}); "
          f"lengthscale {l:.3f} vs {exp_l[k][0]} +- {exp_l[k][1]} (within: {abs(l - exp_l[k][0]) <= exp_l[k][1]})")
d = TAB[(IT, "main")]
print("change of the period from its initial 300 s, by class (median, min, max):",
      d.assign(ch=d.period - 300).groupby("cls").ch.agg(["median", "min", "max"]).round(1).to_dict("index"))

**Result: reproduced, resting on an assumption.** Periods 296.6, 303.8 and 318.0 s and lengthscales 0.724, 0.736 and 0.740, all within the stated spreads; the periods moved from 300 s by medians of -9.9, -4.1 and +1.5 s.

In [ ]:
for key in [(IT, "main"), (IT, "code")]:
    d = TAB[key]
    print(f"--- {key}")
    for k, (m_, s_), seed in [("ON", (0.63, 0.25), 11), ("OFF", (0.36, 0.53), 12), ("NEUTRAL", (-0.9, 0.63), 13)]:
        dk = d[d.cls == k]
        o, p = signflip(dk, "alpha", "subject", seed=seed)
        o2, p2 = signflip(dk, "alpha", "file", seed=seed)
        print(f"  {k}: alpha mean {dk.alpha.mean():.3f} +- {dk.alpha.std():.3f} (paper {m_} +- {s_}; within: {abs(dk.alpha.mean() - m_) <= s_}); "
              f"median {o:.3f}; sign-flip p by animal {p:.4f}, by file {p2:.4f}")

**Result: reproduced, resting on an assumption.** Alpha 0.623 (ON, p 0.0045), 0.342 (OFF, p 0.0104) and -0.786 (NEUTRAL, p below 0.0001), within the stated spreads.

In [ ]:
for key in [(IT, "main"), (IT, "code"), (IT2, "main")]:
    d = TAB[key]
    g = d.groupby("cls").outputscale.agg(["mean", "std", "median"])
    order = g["mean"].sort_values(ascending=False).index.tolist()
    order_med = g["median"].sort_values(ascending=False).index.tolist()
    print(f"{key}: outputscale {g.round(3).to_dict('index')}; order of means {order}, of medians {order_med}")
for k, (m_, s_) in {"ON": (17, 26), "OFF": (3.1, 4), "NEUTRAL": (0.1, 0.2)}.items():
    v = TAB[(IT, "main")].query("cls == @k").outputscale.mean()
    print(f"{k}: mean {v:.3f} vs {m_} +- {s_}: within {abs(v - m_) <= s_}")

**Result: reproduced, resting on an assumption.** Mean outputscale 32.1 (ON) > 5.92 (OFF) > 0.126 (NEUTRAL), each within the stated spread; the medians have the same order.

## fig5d-phase-profiles

> Single-cycle phase-normalized reconstructions of the GP fits show cell-specific oscillatory
> signatures in ON and OFF cells, while NEUTRAL cells have near-constant phase profiles.

The reading named in the criterion: each cell's predicted rate over one fitted period, phase 0 at the
fit's maximum over the training span; depth = (max - min) / mean of that cycle.

In [ ]:
def cell_fit(row):
    return (FIT_A if row.batch == "A" else FIT_B)[IT], int(row.j)

depth, prof = [], {}
for _, row in TAB[(IT, "main")].iterrows():
    res, j = cell_fit(row)
    tmax = res["grid"][j][np.argmax(res["mean_rate"][j])]
    ph = np.linspace(0, 1, 100, endpoint=False)
    r = rvm.gp_predict_rate(res, j, tmax + ph * row.period, seed=7)
    prof[row.ci] = r
    depth.append(dict(cls=row.cls, depth=(r.max() - r.min()) / r.mean(), mean_rate=r.mean() / 5.0))
dep = pd.DataFrame(depth)
print(dep.groupby("cls").depth.describe().round(3))
med = dep.groupby("cls").depth.median()
print(f"NEUTRAL median depth {med['NEUTRAL']:.3f} below ON {med['ON']:.3f} and OFF {med['OFF']:.3f}: "
      f"{med['NEUTRAL'] < med['ON'] and med['NEUTRAL'] < med['OFF']}")
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, k in zip(axs, ["ON", "OFF", "NEUTRAL"]):
    cis = TAB[(IT, "main")].query("cls == @k").ci
    P = np.array([prof[ci] / prof[ci].mean() for ci in cis])
    ax.plot(np.linspace(0, 1, 100, endpoint=False), P.T, lw=0.5)
    ax.set_title(k); ax.set_xlabel("phase (cycles)")
    print(k, "normalized profile range", round(P.min(), 2), "to", round(P.max(), 2))
axs[0].set_ylabel("rate / mean"); plt.tight_layout(); plt.show()

**Result: reproduced, resting on an assumption.** Median modulation depth over one cycle is 1.346 (ON), 1.134 (OFF) and 0.079 (NEUTRAL).

## figs2-phase-latency

> Withdrawal latency shows no systematic relation to the predicted firing rate or oscillatory phase of
> the ON and OFF GP fits at stimulus onset.

As phase_analysis.ipynb: each ON and OFF cell's fit extrapolated to the heat onsets of its whole
recording (times relative to the segment start); predicted rate normalized between the minimum and
maximum of the fit over the training span (cells whose range is 1 count per bin or less dropped, as
the code does); phase from the fitted period with zero at the fit's maximum, as the code's get_phase
(|angle(exp(i (t - t_max) / period))|, no 2 pi; the 2 pi version the notebook's text describes is a
variant); latencies z-scored within cell. Tests named in the criterion: Spearman correlation of
latency with normalized rate, and a circular-linear correlation of latency with phase.

In [ ]:
def circlin(theta, x):
    rxc, rxs = stats.pearsonr(x, np.cos(theta))[0], stats.pearsonr(x, np.sin(theta))[0]
    rcs = stats.pearsonr(np.sin(theta), np.cos(theta))[0]
    R2 = (rxc ** 2 + rxs ** 2 - 2 * rxc * rxs * rcs) / (1 - rcs ** 2)
    return np.sqrt(R2), float(stats.chi2.sf(len(x) * R2, 2))

rows, dropped = [], 0
for _, row in TAB[(IT, "main")].query("cls in ['ON', 'OFF']").iterrows():
    c = cells.loc[row.ci]
    s = sessions[c.session]
    a, b = SEG[row.ci]
    res, j = cell_fit(row)
    rg = res["mean_rate"][j]  # mean count per 5 s bin, as the code's "predicted spikes per bin"
    lo, hi = rg.min(), rg.max()
    if hi - lo <= 1:
        dropped += 1
        continue
    tmax = res["grid"][j][np.argmax(rg)]
    h, wd = rvm.heat(s), rvm.withdrawals(s)
    lat, tt = [], []
    for i, t in enumerate(h):
        nxt = h[i + 1] if i + 1 < len(h) else np.inf
        w_ = wd[(wd > t) & (wd < nxt)]
        if len(w_):
            lat.append(w_[0] - t); tt.append(t - a)
    if len(lat) < 2:
        dropped += 1
        continue
    lat, tt = np.array(lat), np.array(tt)
    pr = rvm.gp_predict_rate(res, j, tt, seed=3)
    z = (lat - lat.mean()) / lat.std()
    for k_ in range(len(tt)):
        rows.append(dict(cls=row.cls, ci=row.ci, file=c.path, onset=round(float(tt[k_] + a), 3), z=z[k_], rate=(pr[k_] - lo) / (hi - lo),
                         phase_code=np.abs(np.angle(np.exp(1j * (tt[k_] - tmax) / row.period))),
                         phase_2pi=np.mod(2 * np.pi * (tt[k_] - tmax) / row.period, 2 * np.pi)))
pl = pd.DataFrame(rows)
print(f"cells used {pl.ci.nunique()} (dropped {dropped}); trials {len(pl)}:", pl.groupby("cls").size().to_dict())
any_sig = False
for k in ["ON", "OFF"]:
    d = pl[pl.cls == k]
    rho, p1 = stats.spearmanr(d.z, d.rate)
    R, p2 = circlin(d.phase_code.values, d.z.values)
    R3, p3 = circlin(d.phase_2pi.values, d.z.values)
    any_sig |= (p1 < 0.05) or (p2 < 0.05)
    print(f"{k}: Spearman latency vs normalized rate rho {rho:.3f}, p {p1:.3f}; circular-linear latency vs phase (code's) "
          f"R {R:.3f}, p {p2:.3f}; with 2 pi: R {R3:.3f}, p {p3:.3f}")
print("any of the four tests significant at 0.05:", any_sig)
# cells of one file share their trials' latencies; variant with one point per (file, heat onset):
# z and the normalized rate averaged over the file's cells of the class (phase tests need one phase per point, so only Spearman)
pf = pl.groupby(["cls", "file", "onset"])[["z", "rate"]].mean().reset_index()
for k in ["ON", "OFF"]:
    d = pf[pf.cls == k]
    rho, p1 = stats.spearmanr(d.z, d.rate)
    print(f"{k}, one point per file and trial ({len(d)} points from {d.file.nunique()} files): Spearman rho {rho:.3f}, p {p1:.3f}")
print(f"normalized rates range {pl.rate.min():.2f} to {pl.rate.max():.2f} (values outside 0 to 1 are onsets outside the fit's training range)")

**Result: reproduced, resting on an assumption.** Over 155 trials of 44 cells no test relates latency to predicted rate (Spearman p 0.85 and 0.090) or phase (circular-linear p 0.56 and 0.59). With one point per file and trial Spearman p is 0.84 and 0.22.

## Summary

| claim | outcome |
|---|---|
| fig5b-train-r2 | reproduced (assumed) |
| fig5b-train-no-difference | not reproduced (assumed) |
| fig5b-test-r2 | reproduced (assumed) |
| fig5b-test-group-differences | reproduced (assumed) |
| table1-hyperparameters | reproduced (assumed) |
| fig5c-dispersion | reproduced (assumed) |
| table1-latent-variance-order | reproduced (assumed) |
| fig5d-phase-profiles | reproduced (assumed) |
| figs2-phase-latency | reproduced (assumed) |

How the analysis differs from the paper's. The code's settings (100 inducing points, p_noise 0.01) for 5000
iterations, the length of the code's saved runs, in place of the Methods' 200 inducing points and p_noise 0.001
and params.json's 10000 iterations, which did not fit the notebook limit. The model is written in plain torch as a
batch of independent GPs. The tests are the code's. The
null model's contamination runs over 0 to the cell's largest count, as the code creates it, and the criterion's
"same likelihood" (0 to 200) is a variant. Matched cell by cell to the code's stored fits by the null model's log
likelihood (median relative difference 0.0002 or less), the fits here agree with them (Spearman r 0.95 to 1.00 for
period, outputscale, alpha and both pseudo-R2), and the stored fits have noise_prob 0.01: the code's settings are
those of the paper's results. The stored NEUTRAL fits of the ongoing files match the fits on the code's 1500 s
segments, not those on whole recordings (median relative difference of the null log likelihoods 0.0003 against
1.28), so the paper used the segments; the criterion's main reading (whole recordings) is the one that finds a
training difference in fig5b-train-no-difference. gp_methods.ipynb runs the Methods' 200 inducing points and
p_noise 0.001.

What was found about the dataset. The fitted hyperparameters and pseudo-R2 values are close to the code's stored
fits, so the dataset holds the data behind Figure 5 and Table 1.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min")